In [1]:
%pip install -Uq langchain langchain_core langchain_groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("안녕하세요!")

history.add_ai_message("안녕하세요! 무엇을 도와드릴까요?")
history.add_user_message("오늘 날씨 어때?")
history.add_ai_message("오늘은 맑고 화창한 날씨입니다. 기온은 25도입니다.")
history.add_user_message("내 이름은 알렉스")
history.add_ai_message("알렉스님, 만나서 반갑습니다! 무엇을 도와드릴까요?")


print(history.messages)

[HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [7]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory 
load_dotenv()

llm = ChatGroq(
    model= "llama-3.1-8b-instant"
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages(
    [
    ("system", "당신은 친절한 비서입니다. 한국어로 대답해주세요."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
    ]
)
chain=prompt | llm | parser

store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_key="input",
    history_key="history"
)






In [8]:
SESSION = "session1"

MESSAGE_HISTORY = [
    "저는 알렉스입니다.",
    "알렉스님, 만나서 반갑습니다! 무엇을 도와드릴까요?"

]

for msg in MESSAGE_HISTORY:
    answer=chain_with_history.invoke(
        {
            "input": msg,
            
        },
        config={"configurable": {"session_id": SESSION}}
    )

msg_count = len(store(SESSION).messages)
print(f"질문: {msg}")

print(f"답변: {answer}")
print(f"총 메시지 수: {msg_count}")

TypeError: Expected mapping type as input to ChatPromptTemplate. Received <class 'list'>.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT 